In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
import lightgbm as lgb


DATA_PATH = "/kaggle/input/competitions/winter-2026-machine-learning-competition"


train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
test_df  = pd.read_csv(f"{DATA_PATH}/test.csv")


X = train_df.drop(columns=["Label"])
y = train_df["Label"]


X_test = test_df.drop(columns=["id"])
test_ids = test_df["id"]


def pauc001(y_true, y_prob):
    return roc_auc_score(y_true, y_prob, max_fpr=0.01)


kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


val_preds_lgb = np.zeros(len(X))
val_preds_rf  = np.zeros(len(X))
val_preds_et  = np.zeros(len(X))
test_preds_lgb = np.zeros(len(X_test))
test_preds_rf  = np.zeros(len(X_test))
test_preds_et  = np.zeros(len(X_test))


for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):

    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

    # Same simple LGB that worked in v54
    model_lgb = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        num_leaves=15,
        verbose=-1,
        random_state=fold,
    )
    model_lgb.fit(X_tr, y_tr)

    val_preds_lgb[val_idx] = model_lgb.predict_proba(X_va)[:, 1]
    test_preds_lgb += model_lgb.predict_proba(X_test)[:, 1] / 5

    # RF1
    rf1 = RandomForestClassifier(
        n_estimators=200,
        max_depth=14,
        min_samples_leaf=2,
        min_samples_split=6,
        max_features="sqrt",
        random_state=fold,
        n_jobs=-1
    )

    # ET
    et = ExtraTreesClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=1,
        min_samples_split=2,
        max_features="sqrt",
        random_state=100 + fold,
        n_jobs=-1
    )

    rf1.fit(X_tr, y_tr)
    et.fit(X_tr, y_tr)

    val_preds_rf[val_idx] = rf1.predict_proba(X_va)[:, 1]
    val_preds_et[val_idx] = et.predict_proba(X_va)[:, 1]
    test_preds_rf += rf1.predict_proba(X_test)[:, 1] / 5
    test_preds_et += et.predict_proba(X_test)[:, 1] / 5


# Print solo scores
print(f"LGB solo:  {pauc001(y, val_preds_lgb):.6f}")
print(f"RF solo:   {pauc001(y, val_preds_rf):.6f}")
print(f"ET solo:   {pauc001(y, val_preds_et):.6f}")
print()


# Search blend weights
print(" Blend search ")
best_score = 0
best_weights = None

for w_rf in np.arange(0.40, 0.85, 0.05):
    for w_lgb in np.arange(0.05, 0.45, 0.05):
        w_et = round(1.0 - w_rf - w_lgb, 2)
        if w_et < 0.0 or w_et > 0.30:
            continue

        blended = w_rf * val_preds_rf + w_et * val_preds_et + w_lgb * val_preds_lgb
        score = pauc001(y, blended)

        if score > best_score:
            best_score = score
            best_weights = (w_rf, w_et, w_lgb)
            print(f"RF={w_rf:.2f} ET={w_et:.2f} LGB={w_lgb:.2f} -> {score:.6f} *")


w_rf, w_et, w_lgb = best_weights
print(f"\nBest: RF={w_rf:.2f} ET={w_et:.2f} LGB={w_lgb:.2f} -> {best_score:.6f}")


test_blend = w_rf * test_preds_rf + w_et * test_preds_et + w_lgb * test_preds_lgb


submission = pd.DataFrame({
    "id": test_ids,
    "Label": test_blend
})


submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved submission.csv")
print(submission.head())

LGB solo:  0.864415
RF solo:   0.917079
ET solo:   0.925918

 Blend search 
RF=0.40 ET=0.30 LGB=0.30 -> 0.919852 *
RF=0.45 ET=0.30 LGB=0.25 -> 0.920144 *
RF=0.50 ET=0.30 LGB=0.20 -> 0.920398 *
RF=0.55 ET=0.30 LGB=0.15 -> 0.920435 *
RF=0.60 ET=0.30 LGB=0.10 -> 0.920686 *
RF=0.65 ET=0.30 LGB=0.05 -> 0.920906 *

Best: RF=0.65 ET=0.30 LGB=0.05 -> 0.920906
Saved submission.csv
   id     Label
0   0  0.001006
1   1  0.000030
2   2  0.000039
3   3  0.000080
4   4  0.000027
